- **SilverWork_incremental_industrial_v2**
- Incremental Silver processing using only new Bronze rows

**Step-1 -Imports and setup**
This cell imports Spark, Winodw, and Delta helpers, switched to the right catlaog, makes sure the Silver schema exists and creates a silver_run_id for the current run.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog datatocrunch_novacart_adb")
spark.sql("create schema if not exists silver_schema")

silver_run_id = str(uuid.uuid4())
print("Current Silver Run ID: ", silver_run_id)

Step 2 _ Silver control table
This table stores the latest Silver processing state for each entity.

It helps us track:
- the latest Brinze run already processed by silver
- the latest bronze ingestion timestamp already processed
- how many row were merged in the latest Silver run

In [0]:
spark.sql("""
          create table if not exists datatocrunch_novacart_adb.silver_schema.processing_control(
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
          )
          using delta
          """)

step-3 --Helper functions 
This cell contains reusable logic for silver:
-   upsert_to_silver() merges cleaned/transformed rows into the silver target table
-   get_last_processed_bronze_ingested_at() reads the Silver watermark
-   upsert_silver_control() updates the Silver control table
-   get_incremental_bronze() reads only new bronze rows that silver has not processed yet

In [0]:
def upsert_to_silver(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
         .merge(df_source.alias("source"),f"target.{join_key} = source.{join_key}")
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute()
         )
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
    ctrl = (
        spark.table("datatocrunch_novacart_adb.silver_schema.processing_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    rows = ctrl.collect()
    if not rows:     #if rows is empty:
        return None, None

    print(rows[0]["last_processed_bronze_ingested_at"])
    print(rows[0]["last_processed_bronze_run_id"])
    
    return( rows[0]["last_processed_bronze_ingested_at"], rows[0]["last_processed_bronze_run_id"] )

  

In [0]:
def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                rows_merged,
                "SUCCESS",
                silver_run_id,
                datetime.utcnow()            
            )
        ],
        schema = """
        layer string,
        entity_name string,
        last_processed_bronze_run_id string,
        last_processed_bronze_ingested_at timestamp,
        rows_merged bigint,
        run_status string,
        silver_run_id string,
        updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "datatocrunch_novacart_adb.silver_schema.processing_control")
    (dt.alias("t")
     .merge(ctrl_df.alias("s"), "t.layer = s.layer and t.entity_name = s.entity_name")
     .whenMatchedUpdate(set={
         "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
         "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
         "rows_merged": "s.rows_merged",
         "run_status": "s.run_status",
         "silver_run_id": "s.silver_run_id",
         "updated_at": "s.updated_at"
     })
     .whenNotMatchedInsertAll()
     .execute()
     )
    

In [0]:
def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at, last_run_id = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at
    return bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at)), last_ingested_at

step -4 Orders Incremental Processing

This cell processed orders form Bronze to Silver

It does the following:
-  reads only new bronze order rows
- cleans values like order_status and order_amount
-  keeps only the latest version per order_id
-  validate business rules
-  sends bad rows to quarantine
-  merges good rows into orders_transformed

In [0]:
df_raw = spark.sql("select * from datatocrunch_novacart_adb.bronze_schema.orders_raw")
display(df_raw)

In [0]:
#Step -4 -Orders incremental processing
#Read only the Bronze order rows that Silver has not processed yet.

orders_inc , last_orders_ingested_at = get_incremental_bronze(
    "datatocrunch_novacart_adb.bronze_schema.orders_raw", "orders")

#count the incremental order rows entering silver in this run.
orders_inc_count = orders_inc.count()

#only run silver order cleaning and validataion when there are new bronze order rows.

if orders_inc_count >0:
    #Create a window that keeps the latest order record for each order_id.
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    #Start the Silver order-cleaning pipeline.This block standardizes and deduplicates raw order records.
    orders_cleaned = (
        orders_inc
        #Standardize order_status to uppercase so values such as shipped and SHIPPED become consistent.
        .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
        .withColumn("order_status", F.when(F.col("order_status")=="", F.lit(None)).otherwise(F.col("order_status")))
        #Remove formating characters from order_amount so it can be cast to a numberic type.
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"[$, ]", ""))
        .withColumn("order_amount", F.when(F.trim(F.col("order_amount")).isin("N/A", "NULL","??"),None).otherwise(F.col("order_amount")))
        .withColumn("order_amount",F.col("order_amount").cast("double"))
        .withColumn("updated_at",F.to_timestamp("updated_at"))

        #Assign a row number inside each business key so we can kepp only the latest version of that record.
        .withColumn("row_rank",F.row_number().over(order_window))
        #keep only the latest record for each business key
        .filter(F.col("row_rank")==1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    #Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        orders_cleaned,
        "datatocrunch_novacart_adb.silver_schema.orders_cleaned",
        "order_id"    
    )

    #Apply Silver data-quality rules to the cleaned order records
    orders_validated = (
        orders_cleaned
        .withColumn(
            "to_be_verified_by_orders_team",
            F.when(F.col("customer_id").isNull(), "verify_customer_id")
            .when(F.col("product_id").isNull(), "verify_product_id")
            .when((F.col("order_status").isNull()) | (F.trim(F.col("order_status")) == ""), "verify_order_status")
            .when((F.col("order_amount").isNull()) | (F.col("order_amount") <= 0), "verify_order_amount")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_order_amount", F.when((F.col("order_amount").isNull()) | (F.col("order_amount") <=0), F.lit(True)).otherwise(F.lit(False))
        )
        .withColumn("order_date", F.to_date("created_at"))
        .withColumn("order_year", F.year("created_at"))
        .withColumn("order_month",F.month("created_at"))
        .withColumn("order_day", F.dayofmonth("created_at"))
        .withColumn("order_row", F.date_format("created_at", "E"))
    )
    #keep only valid rows for the transformed Silver table.
    orders_good = orders_validated.filter(F.col("to_be_verified_by_orders_team") == "No Issues")

    #send invalid order rows to the quarnatine dataset for manual review>
    orders_bad = (
        orders_validated
        .filter(F.col("to_be_verified_by_orders_team") != "No Issuses")
        .withColumn("quarantine_ts", F.current_timestamp())    
    )
    #Merge the cleaned or validated Sivler dataset into its delta target table.
    upsert_to_silver(
        orders_good,
        "datatocrunch_novacart_adb.silver_schema.orders_transformed",
        "order_id"
    )
    # Append bad records rows to the quarantize table instead of losing them.
    orders_bad.write.format("delta").mode("append").saveAsTable("datatocrunch_novacart_adb.silver_schema.orders_quarantine")

    mx_ingested = orders_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        orders_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias('mx'))
        .collect()[0]["mx"]
    )
    upsert_silver_control("orders", mx_run, mx_ingested, orders_good.count())

else:
    print("No new orders Bronze rows for Silver.")
    upsert_silver_control(
        "orders",
        None,
        last_orders_ingested_at,
        orders_inc_count
    )




In [0]:
%sql
select * from datatocrunch_novacart_adb.silver_schema.orders_cleaned;

In [0]:
%sql
select * from datatocrunch_novacart_adb.silver_schema.orders_transformed;

In [0]:
%sql
select * from datatocrunch_novacart_adb.silver_schema.orders_quarantine;

**Step 5 - Products incremental processing**
This cell processes Products from Bronze to Silver.

It handles:
- product name cleanup
- category standardization
- price cleanup and numeric conversion
- latest record selection per product_id
- data quality validation
- quarantine for bad rows
- merge into silver current-state tables 

In [0]:
#Step 5 - Products incremental processing
#Read only the Bronze product rows that silver has not processed yet.

products_inc, last_products_ingestion_at = get_incremental_bronze("datatocrunch_novacart_adb.bronze_schema.products_raw", "products")

#Count the incremental product rows entering silver in this run.

products_inc_count = products_inc.count()
print(f"products rows_to_process_in_silver = {products_inc_count}")

if products_inc.count() > 0:
    # create a window that keeps the latest product record for each product_id.

    product_window = Window.partitionBy("product_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # Start the silver product-cleaning pipeline. This block standardizes and deduplicates raw product records.
    products_cleaned = (
        products_inc
        #standardize product_name by trimming spaces and converting text to uppercase.
        .withColumn("product_name", F.upper(F.trim(F.col("product_name"))))
        .withColumn("product_name", F.when(F.col("product_name") == "", F.lit(None)).otherwise(F.col("product_name")))
        .withColumn(
            "category",
            F.when(F.upper(F.trim(F.col('category'))).contains("ELECTRNICS"), "ELECTRONICS")
            .otherwise(F.upper(F.trim(F.col("category"))))
        )
        # start cleaning the product process field before converting it to numeric.
        .withColumn("price",F.trim(F.col("price")))
        .withColumn("price",F.regexp_replace(F.col("price"), r"\$", ""))
        .withColumn("price",F.regexp_replace(F.col("price"), ",","."))
        .withColumn("price",F.regexp_replace(F.col("price"), r"\s+", ""))
        .withColumn("price",F.expr("try_cast(price as double)"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        #Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", F.row_number().over(product_window))
        #Keep only the latest record for each business key.
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))             
    )

    #Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(products_cleaned,"datatocrunch_novacart_adb.silver_schema.products_cleaned","product_id")

    #Apply Silver data-quality rules to the cleaned product records.
    products_validated = (
        products_cleaned 
        .withColumn(
            "to_be_verified_by_product_team",
            F.when(F.col("product_name").isNull(), "verify_product_name")
            .when(F.col("category").isNull(), "verify_category")
            .when(F.col("price").isNull() | (F.col("price") <= 0), "verify_price")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_product_price",
            F.when(F.col("price").isNull() | (F.col("price") <= 0), "invalid_price").otherwise("valid_price")
        )
    )
    #keep only valid product rows for the transformed Silver table
    products_good = products_validated.filter(
        (F.col("to_be_verified_by_product_team") == "No Issues") &
        (F.col("check_product_price") == "valid_price")         
    )
    if "price_raw" in products_good.columns:
        #Keep only valid product rows for the transformed Silver table.
        products_good = products_good.drop("price_raw")
    
    #send invalid product rows to the dataset for manual review
    products_bad = products_validated.filter(
        (F.col("to_be_verified_by_product_team") != "No Issues") |
        (F.col("check_product_price") == "invalid_price")
    ).withColumn("quarantine_ts", F.current_timestamp())
    
    #Merge the cleaned or validated Silver dataset into its Delta target tabke.
    upsert_to_silver(products_good,"datatocrunch_novacart_adb.silver_schema.products_good","product_id")
    
    #Append bad product rows to the quarantine table instead of losing them.
    products_bad.write.format("delta").mode("append").saveAsTable("datatocrunch_novacart_adb.silver_schema.products_quarantine")

    mx_ingested =products_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = products_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]

    upsert_silver_control("products", mx_run,mx_ingested, products_good.count())
else:
    print("No new products Bronze rows for silver.")
    upsert_silver_control(
        "products",
        None,
        last_products_ingestion_at,
        products_inc_count
    )